# Stage 4 — Data-Size Scaling of DFR on Waterbirds

**Paper:** Kirichenko, Izmailov & Wilson, *Last Layer Re-Training is Sufficient for Robustness to Spurious Correlations*, ICLR 2023.

**Experiment type:** E. Data-size scaling.

**Question:** How does DFR's worst-group test accuracy change as the number of group-balanced examples per group, `k`, increases — and where does it plateau?

This notebook does **not** retrain the CNN. It reuses the frozen features saved by Stage 3:

```text
results/dfr_features.npz
```

The test set is used only for final evaluation, never for tuning.


## Design

| Item | Choice |
|---|---|
| Independent variable | `k` = balanced examples per group |
| `k` values | `1, 2, 4, 8, 16, 32, 64, 128, 133` |
| Total balanced examples | `4k` |
| Primary metric | Worst-group test accuracy |
| Secondary metrics | Overall and per-group test accuracy |
| Control | ERM at 79.0% worst-group |
| Head | L1 logistic regression, fixed `C = 0.1` |
| Retrains per run | 10, averaged weights |
| Seeds | 5 |
| Test set | Evaluated once per run |

### What `C` is

The DFR head is a **logistic regression**: a linear classifier on the 2048 frozen CNN features. `C` is scikit-learn's **inverse L1 regularization strength**.

- **Small `C`** = strong regularization: weights are pushed toward zero, and many are set exactly to zero.
- **Large `C`** = weak regularization: the head can use more features and fit the balanced data more aggressively.

With 2048 features and only a few hundred examples, regularization is essential. Stage 3 tuned `C` on validation and selected `C = 0.1` (tied with 0.07). Here we keep it fixed so the only changing variable is `k`.

**Limitation:** the optimal `C` might differ at very small `k`. We will note this in the report; the main experiment deliberately fixes it to isolate data size.


In [ ]:
# This is a new Colab notebook, so Drive must be mounted explicitly.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import json
import os
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

PROJECT_DIR = Path("/content/drive/MyDrive/dfr_waterbirds_project")
RESULTS_DIR = PROJECT_DIR / "results"
OUT_DIR = RESULTS_DIR / "stage4"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_PATH = RESULTS_DIR / "dfr_features.npz"
STAGE3_RESULTS = RESULTS_DIR / "dfr_results.json"

if not FEATURES_PATH.exists():
    print("Drive root exists: ", os.path.isdir("/content/drive/MyDrive"))
    print("Project dir exists:", PROJECT_DIR.exists())
    if PROJECT_DIR.exists():
        print("Project dir contents:", os.listdir(PROJECT_DIR))
    if RESULTS_DIR.exists():
        print("Results dir contents:", os.listdir(RESULTS_DIR))
    raise FileNotFoundError(
        f"Missing {FEATURES_PATH}. Mount Drive and run Stage 3 first."
    )

data = np.load(FEATURES_PATH)
val_X, val_y, val_g = data["val_X"], data["val_y"], data["val_g"]
test_X, test_y, test_g = data["test_X"], data["test_y"], data["test_g"]

GROUP_NAMES = [
    "landbird / land",
    "landbird / water",
    "waterbird / land",
    "waterbird / water",
]

print("val features: ", val_X.shape)
print("test features:", test_X.shape)
print("val groups:  ", np.bincount(val_g, minlength=4))
print("test groups: ", np.bincount(test_g, minlength=4))

assert val_X.shape[1] == 2048
assert test_X.shape[1] == 2048
assert all((val_g == g).sum() > 0 for g in range(4))
assert all((test_g == g).sum() > 0 for g in range(4))

# Stage 3 metrics, for the ERM baseline line and the k=133 sanity anchor.
if STAGE3_RESULTS.exists():
    with open(STAGE3_RESULTS) as f:
        stage3 = json.load(f)
    erm_overall = float(stage3["erm"]["overall"])
    erm_worst = float(stage3["erm"]["worst_group"])
    stage3_dfr_overall = float(stage3["dfr"]["overall"])
    stage3_dfr_worst = float(stage3["dfr"]["worst_group"])
    stage3_k = int(stage3.get("n_per_group", 133))
else:
    print("WARNING: dfr_results.json not found; using Stage 3 numbers from the notebook output.")
    erm_overall, erm_worst = 0.894, 0.790
    stage3_dfr_overall, stage3_dfr_worst = 0.931, 0.909
    stage3_k = 133

print(f"ERM baseline:      overall={erm_overall:.3f}, worst-group={erm_worst:.3f}")
print(f"Stage 3 DFR (k={stage3_k}): overall={stage3_dfr_overall:.3f}, worst-group={stage3_dfr_worst:.3f}")

C_FIXED = 0.1
N_SEEDS = 5
N_RETRAINS = 10
K_GRID = [1, 2, 4, 8, 16, 32, 64, 128, 133]

max_k = int(np.bincount(val_g, minlength=4).min())
k_grid = [k for k in K_GRID if k <= max_k]
if max_k not in k_grid:
    k_grid.append(max_k)
k_grid = sorted(set(k_grid))

print("max k per group:", max_k)
print("k grid:", k_grid)
print("total examples per k:", [4 * k for k in k_grid])
print("fixed C:", C_FIXED)

# Standardize once on the full validation set; this is unsupervised and fixed
# across all k so preprocessing is not a changing variable.
scaler = StandardScaler().fit(val_X)
val_Xs = scaler.transform(val_X)
test_Xs = scaler.transform(test_X)


In [ ]:
def group_metrics(y_true, y_pred, groups):
    per_group = np.array([
        float((y_pred[groups == g] == y_true[groups == g]).mean())
        for g in range(4)
    ])
    return {
        "overall": float((y_pred == y_true).mean()),
        "worst_group": float(per_group.min()),
        "per_group": per_group,
    }


def make_group_perms(groups, n_retrains, seed):
    """For each retrain, a separate random permutation per group."""
    rng = np.random.default_rng(seed)
    return [
        [rng.permutation(np.flatnonzero(groups == g)) for g in range(4)]
        for _ in range(n_retrains)
    ]


rows = []
start_time = time.time()

for seed in range(N_SEEDS):
    perms = make_group_perms(val_g, N_RETRAINS, seed=seed)

    for k in k_grid:
        coefs, intercepts = [], []

        for r in range(N_RETRAINS):
            idx = np.concatenate([perms[r][g][:k] for g in range(4)])

            clf = LogisticRegression(
                penalty="l1",
                solver="liblinear",
                C=C_FIXED,
                max_iter=2000,
            )
            clf.fit(val_Xs[idx], val_y[idx])
            coefs.append(clf.coef_[0])
            intercepts.append(clf.intercept_[0])

        coef = np.mean(coefs, axis=0)
        intercept = float(np.mean(intercepts))

        pred = (test_Xs @ coef + intercept > 0).astype(int)
        m = group_metrics(test_y, pred, test_g)

        row = {
            "seed": seed,
            "k": int(k),
            "n_total": int(4 * k),
            "C": C_FIXED,
            "n_retrains": N_RETRAINS,
            "overall": m["overall"],
            "worst_group": m["worst_group"],
        }
        row.update({f"group_{i}": m["per_group"][i] for i in range(4)})
        rows.append(row)

        print(
            f"seed {seed + 1}/{N_SEEDS} | k={k:3d} | "
            f"worst={m['worst_group']:.3f} | overall={m['overall']:.3f}"
        )

df = pd.DataFrame(rows)
print(f"\nFinished {len(df)} runs in {time.time() - start_time:.1f}s")


In [ ]:
summary = (
    df.groupby("k")
    .agg(
        n_total=("n_total", "first"),
        C=("C", "first"),
        n_retrains=("n_retrains", "first"),
        worst_mean=("worst_group", "mean"),
        worst_std=("worst_group", "std"),
        overall_mean=("overall", "mean"),
        overall_std=("overall", "std"),
        group_0_mean=("group_0", "mean"),
        group_1_mean=("group_1", "mean"),
        group_2_mean=("group_2", "mean"),
        group_3_mean=("group_3", "mean"),
    )
    .reset_index()
)

summary["delta_worst_vs_erm"] = summary["worst_mean"] - erm_worst
summary["delta_worst_vs_stage3"] = summary["worst_mean"] - stage3_dfr_worst

best_worst = float(summary["worst_mean"].max())
best_row = summary.loc[summary["worst_mean"].idxmax()]

# Plateau rule: smallest k after which every point is within 1 percentage point
# of the best mean worst-group accuracy.
plateau_k = None
for i in range(len(summary)):
    if (summary.iloc[i:]["worst_mean"] >= best_worst - 0.01).all():
        plateau_k = int(summary.iloc[i]["k"])
        break

df.to_csv(OUT_DIR / "stage4_runs.csv", index=False)
summary.to_csv(OUT_DIR / "stage4_summary.csv", index=False)
summary.to_json(OUT_DIR / "stage4_summary.json", orient="records", indent=2)

stage4 = {
    "question": "How does DFR worst-group accuracy change with group-balanced examples per group?",
    "experiment_type": "E. Data-size scaling",
    "C": C_FIXED,
    "n_seeds": N_SEEDS,
    "n_retrains": N_RETRAINS,
    "k_grid": k_grid,
    "erm": {"overall": erm_overall, "worst_group": erm_worst},
    "stage3_dfr": {
        "k": stage3_k,
        "overall": stage3_dfr_overall,
        "worst_group": stage3_dfr_worst,
    },
    "best_worst_mean": best_worst,
    "best_k": int(best_row["k"]),
    "plateau_k": plateau_k,
    "plateau_rule": "smallest k after which all points are within 1 percentage point of the best mean",
    "summary": json.loads(summary.to_json(orient="records")),
}
with open(OUT_DIR / "stage4_results.json", "w") as f:
    json.dump(stage4, f, indent=2)

print("Summary")
print(summary[[
    "k", "n_total", "worst_mean", "worst_std",
    "overall_mean", "overall_std", "delta_worst_vs_erm",
]].to_string(index=False))

print(f"\nBest mean worst-group: {best_worst:.3f} at k={int(best_row['k'])}")
if plateau_k is None:
    print("Plateau: not reached within the tested k range.")
else:
    print(f"Plateau: k={plateau_k} ({4 * plateau_k} balanced examples)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

panels = [
    (axes[0], "worst_mean", "worst_std", erm_worst, "Worst-group accuracy", "Worst-group test accuracy"),
    (axes[1], "overall_mean", "overall_std", erm_overall, "Overall accuracy", "Overall test accuracy"),
]

for ax, mean_col, std_col, baseline, legend, ylabel in panels:
    ax.errorbar(
        summary["k"],
        summary[mean_col],
        yerr=summary[std_col].fillna(0),
        marker="o",
        capsize=3,
        linewidth=1.5,
        label="DFR (mean ± std)",
    )
    ax.axhline(baseline, color="black", linestyle="--", linewidth=1.2,
               label=f"ERM ({baseline:.1%})")
    ax.set_xscale("log", base=2)
    ticks = [k for k in summary["k"] if k != 133]
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(int(k)) for k in ticks])
    ax.set_xlim(0.8, 150)
    ax.set_xlabel("Balanced examples per group, k")
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

if plateau_k is not None:
    for ax in axes:
        ax.axvline(plateau_k, color="green", linestyle=":", linewidth=1,
                   label=f"Plateau k={plateau_k}")

fig.suptitle("Waterbirds: data-size scaling of last-layer re-training")
fig.tight_layout()
fig.savefig(OUT_DIR / "stage4_scaling.png", dpi=200)
plt.show()

# Per-group accuracy vs k.
fig2, ax = plt.subplots(figsize=(6.8, 4.2))
for i, name in enumerate(GROUP_NAMES):
    ax.plot(summary["k"], summary[f"group_{i}_mean"], marker="o", label=name)

ax.set_xscale("log", base=2)
ticks = [k for k in summary["k"] if k != 133]
ax.set_xticks(ticks)
ax.set_xticklabels([str(int(k)) for k in ticks])
ax.set_xlim(0.8, 150)
ax.set_xlabel("Balanced examples per group, k")
ax.set_ylabel("Test accuracy")
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
fig2.tight_layout()
fig2.savefig(OUT_DIR / "stage4_per_group.png", dpi=200)
plt.show()

print("Saved:", OUT_DIR / "stage4_scaling.png")
print("Saved:", OUT_DIR / "stage4_per_group.png")


In [ ]:
first = summary.iloc[0]
last = summary.iloc[-1]

print("EXPERIMENT SUMMARY")
print("=" * 60)
print(f"Balanced examples per group: {int(first['k'])} to {int(last['k'])}")
print(f"Total balanced examples:     {int(first['n_total'])} to {int(last['n_total'])}")
print(f"Worst-group at k={int(first['k'])}: {first['worst_mean']:.1%}")
print(f"Worst-group at k={int(last['k'])}: {last['worst_mean']:.1%}")
print(f"Best mean worst-group:      {best_worst:.1%} at k={int(best_row['k'])}")
print(f"ERM baseline:               {erm_worst:.1%}")
print(f"Stage 3 DFR (k={stage3_k}):     {stage3_dfr_worst:.1%}")

if plateau_k is None:
    print("\nNo plateau within the tested range; worst-group accuracy is still improving.")
else:
    print(f"\nPlateau: k={plateau_k} ({4 * plateau_k} balanced examples).")
    print("This is the smallest k whose mean is within 1 percentage point of the best mean,")
    print("with no larger k improving by more than 1 point.")

print("\nTwo-sentence summary for a teammate:")
print(
    f"We varied the number of group-balanced examples per group from {int(first['k'])} to "
    f"{int(last['k'])} while keeping the frozen ERM features and C={C_FIXED} fixed. "
    f"Worst-group test accuracy moved from {first['worst_mean']:.1%} at k={int(first['k'])} "
    f"to {last['worst_mean']:.1%} at k={int(last['k'])}, with a best mean of "
    f"{best_worst:.1%} at k={int(best_row['k'])}."
)
if plateau_k is not None:
    print(
        f"The curve reached within 1 point of its best value at k={plateau_k} "
        f"({4 * plateau_k} balanced examples), so that is the practical data budget."
    )


## What to report

- **Main figure:** `stage4_scaling.png` — worst-group accuracy vs `k`, with the ERM line.
- **Secondary figure:** `stage4_per_group.png` — which groups improve or drop.
- **Table:** `stage4_summary.csv` — `k`, total examples, worst-group mean/std, overall mean/std.
- **Raw runs:** `stage4_runs.csv` — one row per seed and `k`.
- **Metadata:** `stage4_results.json` — protocol, ERM baseline, Stage 3 anchor, plateau rule.

### Key caveats

- `C` is fixed at 0.1, selected by Stage 3 on validation. The optimal `C` may differ at small `k`.
- The CNN is not retrained. All runs share the same frozen features.
- The test set is used only for final evaluation.
- `k = 1` has only four balanced examples, so its variance is expected to be high.
- At very small `k`, L1 with fixed `C = 0.1` can shrink all coefficients to zero. That is a genuine failure mode of the fixed-`C` recipe, not a bug; report it rather than hiding it.
